# Copyright

<PRE>
Jelen iPython notebook a Budapesti Műszaki és Gazdaságtudományi Egyetemen tartott
"MI ágensek és multiágensrendszerek fejlesztése" tantárgy segédanyagaként készült.

A notebook bármely részének újra felhasználása, publikálása csak a szerzők írásos beleegyezése esetén megegengedett.

2026 (c) Potyók Csaba (potyok kukac mit pont bme pont hu)
</PRE>

# MCP eszközök fejlesztése

A gyakorlat betekintést nyújt abba, hogyan lehet MCP szervereket készíteni, amelyeket az LLM-alapú ágensek fel tudnak használni a probléma megoldásuk során. A gyakorlat során megnézzük, hogy lehet MCP szerverekhez eszközöket létrehozni.

# Szükséges könyvtárak

A gyakorlat során a [Pydantic AI](https://ai.pydantic.dev/) könyvtárat használjuk, mivel ennek segítségével típusbiztosan és egyszerűen tudunk eszközöket definiálni.

In [ ]:
#!pip install pydantic-ai sqlmodel "pydantic-ai-slim[fastmcp]" --quiet
%pip install pydantic-ai sqlmodel "pydantic-ai-slim[fastmcp]" fastmcp logfire ipython sqlalchemy pywin32 --quiet

# Környezeti változók beállítása

Az API kulcs biztonságos kezelése érdekében a Google Colab `Titkos kulcsok` funkcióját használjuk, ahol biztonságosan hozzáférhetünk a notebook futtatása során a használt API kulcsainkhoz.

In [ ]:
import os
from getpass import getpass

mistral_api_key = os.getenv('MISTRAL_API_KEY') or os.getenv('Mistral_API_KEY')
if not mistral_api_key:
    mistral_api_key = getpass('Enter your Mistral API key: ')

os.environ['MISTRAL_API_KEY'] = mistral_api_key
os.environ['Mistral_API_KEY'] = mistral_api_key

# 1. Naptárbejegyzéseket kezelő MCP szerver

Ebben a szakaszban készítünk egy egyszerű MCP szervert, amely képes kiolvasni bejegyzéseket a felhasználó naptárából. A naptár elkészítéséhez a `sqlmodel` modult fogjuk használni annak érdekében, hogy egy valódi adatbázist tudjuk Colab környezetben szimulálni.

A naptárbejegyzéseket tartalmazó adatbázis előkészítése.

In [ ]:
from sqlmodel import SQLModel, Field, create_engine, Session, select, JSON, Column, Relationship, or_, and_
from sqlalchemy.orm import selectinload
from typing import List, Optional, Literal, Union
from datetime import datetime, timedelta

Definiáljuk a `CalendarEntry` adatmodellt, amely tartalmazza a következő mezőket: `title`, `description`, `start_time` és `end_time`. Adjunk meg minden mezőhöz egy rövid leírást. Az időpontok kezeléséhez használjuk a `datetime` modult.

In [ ]:
class CalendarEntry(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    title: str = Field(description="Event title")
    description: str = Field(description="Event description")
    start_time: datetime = Field(description="Event start time")
    end_time: datetime = Field(description="Event end time")

Inicializáljuk az adatbázist!

In [ ]:
calendar_sqlite_url = "sqlite:///calendar.db"
calendar_engine = create_engine(calendar_sqlite_url)

In [ ]:
SQLModel.metadata.create_all(calendar_engine)

In [ ]:
# Ha összeakadnának a tábla verziók, ezzel lehet törölni a sémákat
# SQLModel.metadata.clear()

## Példa adatok

Adjunk meg néhány példa bejegyzést, amelyet a naptárunk fog tartalmazni!

In [ ]:
with Session(calendar_engine) as session:
    example_events = [
        CalendarEntry(
            title="Project Kickoff",
            description="Initial meeting for the new MCP tutorial series.",
            start_time=datetime(2026, 1, 20, 9, 0),
            end_time=datetime(2026, 1, 20, 11, 0)
        ),
        CalendarEntry(
            title="Quick Sync",
            description="Daily stand-up with the development team.",
            start_time=datetime(2026, 1, 21, 10, 0),
            end_time=datetime(2026, 1, 21, 10, 15)
        ),
        CalendarEntry(
            title="Deep Work: Coding",
            description="Implementing the Pydantic AI agent logic.",
            start_time=datetime(2026, 1, 23, 13, 0),
            end_time=datetime(2026, 1, 23, 17, 0)
        ),
        CalendarEntry(
            title="Client Feedback Session",
            description="Reviewing the first draft of the database schema.",
            start_time=datetime(2026, 1, 26, 15, 0),
            end_time=datetime(2026, 1, 26, 16, 30)
        ),
        CalendarEntry(
            title="Gym Workout",
            description="Morning strength training session.",
            start_time=datetime(2026, 1, 29, 8, 0),
            end_time=datetime(2026, 1, 29, 9, 15)
        ),
        CalendarEntry(
            title="Strategy Planning",
            description="Quarterly goals and roadmap definition.",
            start_time=datetime(2026, 2, 2, 10, 0),
            end_time=datetime(2026, 2, 2, 14, 0)
        ),
        CalendarEntry(
            title="Technical Interview",
            description="Hiring interview for the Junior Dev position.",
            start_time=datetime(2026, 2, 5, 11, 0),
            end_time=datetime(2026, 2, 5, 12, 0)
        ),
        CalendarEntry(
            title="Lunch with Mentor",
            description="Discussing career growth and MCP integration.",
            start_time=datetime(2026, 2, 9, 12, 30),
            end_time=datetime(2026, 2, 9, 14, 0)
        ),
        CalendarEntry(
            title="Bug Scrub",
            description="Reviewing open issues in the GitHub repository.",
            start_time=datetime(2026, 2, 12, 16, 0),
            end_time=datetime(2026, 2, 12, 16, 45)
        ),
        CalendarEntry(
            title="Valentine's Dinner",
            description="Private dinner reservation at the Italian bistro.",
            start_time=datetime(2026, 2, 14, 19, 0),
            end_time=datetime(2026, 2, 14, 21, 30)
        ),
        CalendarEntry(
            title="Architecture Review",
            description="Evaluating the system design with senior leads.",
            start_time=datetime(2026, 2, 17, 14, 0),
            end_time=datetime(2026, 2, 17, 16, 0)
        ),
        CalendarEntry(
            title="Coffee Break Sync",
            description="Casual chat with the design team.",
            start_time=datetime(2026, 2, 20, 10, 30),
            end_time=datetime(2026, 2, 20, 10, 45)
        ),
        CalendarEntry(
            title="Workshop: SQLModel",
            description="Internal training on using SQLModel for AI apps.",
            start_time=datetime(2026, 2, 24, 9, 0),
            end_time=datetime(2026, 2, 24, 12, 0)
        ),
        CalendarEntry(
            title="Market Research",
            description="Analyzing competitors in the AI agent space.",
            start_time=datetime(2026, 2, 27, 15, 0),
            end_time=datetime(2026, 2, 27, 17, 45)
        ),
        CalendarEntry(
            title="Code Review",
            description="Peer review of the latest pull requests.",
            start_time=datetime(2026, 3, 2, 11, 0),
            end_time=datetime(2026, 3, 2, 11, 45)
        ),
        CalendarEntry(
            title="Product Demo",
            description="Showcasing the MCP server to stakeholders.",
            start_time=datetime(2026, 3, 5, 14, 0),
            end_time=datetime(2026, 3, 5, 15, 0)
        ),
        CalendarEntry(
            title="Language Lesson",
            description="Weekly advanced English conversation class.",
            start_time=datetime(2026, 3, 9, 18, 0),
            end_time=datetime(2026, 3, 9, 19, 0)
        ),
        CalendarEntry(
            title="Sprint Planning",
            description="Task allocation for the upcoming two weeks.",
            start_time=datetime(2026, 3, 11, 9, 0),
            end_time=datetime(2026, 3, 11, 12, 0)
        ),
        CalendarEntry(
            title="Status Update",
            description="Quick call to update management on progress.",
            start_time=datetime(2026, 3, 13, 16, 0),
            end_time=datetime(2026, 3, 13, 16, 15)
        ),
        CalendarEntry(
            title="Final Testing",
            description="End-to-end testing of the Colab environment.",
            start_time=datetime(2026, 3, 15, 10, 0),
            end_time=datetime(2026, 3, 15, 14, 0)
        )
    ]
    session.add_all(example_events)
    session.commit()

## MCP szerver elkészítése

Importáljuk a `pydantic_ai` modult, amelynek segítségével hozzuk létre az MCP szerverünket, majd az ágensünket!

In [ ]:
from fastmcp import FastMCP

from pydantic_ai.toolsets.fastmcp import FastMCPToolset

Definiáljuk a `Calendar` nevű MCP szervert, továbbá adjuk hozzá a következő eszközöket, amelyeket az ágensünket fog majd használni!

In [ ]:
calendar_mcp_server = FastMCP('Calendar')

Definiáljuk a `get_today` eszközt, amelynek a segítségével egy beégett dátumot kapunk vissza, amely a mai nap dátumaként fog szolgálni!

In [ ]:
@calendar_mcp_server.tool()
def get_today():
  return datetime.now().date().isoformat()

Definiáljuk a `get_events_for_week` eszközt, amely képes egy adott `target_date` hetére eső összes naplóbejegyzést kilistázni.

In [ ]:
@calendar_mcp_server.tool()
def get_events_for_week(target_date: datetime):
    start_of_week = (target_date - timedelta(days=target_date.weekday())).replace(hour=0, minute=0, second=0, microsecond=0)
    end_of_week = start_of_week + timedelta(days=7)

    with Session(calendar_engine) as session:
        statement = select(CalendarEntry).where(
            and_(
                CalendarEntry.start_time >= start_of_week,
                CalendarEntry.start_time < end_of_week,
            )
        ).order_by(CalendarEntry.start_time)
        events = session.exec(statement).all()

    return [
        {
            "id": event.id,
            "title": event.title,
            "description": event.description,
            "start_time": event.start_time.isoformat(),
            "end_time": event.end_time.isoformat(),
        }
        for event in events
    ]

Definiáljuk a `get_events_after_date` eszközt, amely egy adott `target_date` dátumhoz képest `days_offset` napra lévő eseményeket tudja kilistázni.

In [ ]:
@calendar_mcp_server.tool()
def get_events_after_date(target_date: datetime, days_offset: int = 0):
  target_day = (target_date + timedelta(days=days_offset)).replace(hour=0, minute=0, second=0, microsecond=0)
  next_day = target_day + timedelta(days=1)

  with Session(calendar_engine) as session:
      statement = select(CalendarEntry).where(
          and_(
              CalendarEntry.start_time >= target_day,
              CalendarEntry.start_time < next_day,
          )
      ).order_by(CalendarEntry.start_time)
      events = session.exec(statement).all()

  return [
      {
          "id": event.id,
          "title": event.title,
          "description": event.description,
          "start_time": event.start_time.isoformat(),
          "end_time": event.end_time.isoformat(),
      }
      for event in events
  ]

Definiáljuk a `insert_event` eszközt, amivel az ágens fel tud venni egy új naptárbejegyzést!

In [ ]:
@calendar_mcp_server.tool()
def insert_event(title: str, description: str, start_time: datetime, end_time: datetime):
  with Session(calendar_engine) as session:
      event = CalendarEntry(
          title=title,
          description=description,
          start_time=start_time,
          end_time=end_time,
      )
      session.add(event)
      session.commit()
      session.refresh(event)

  return {
      "id": event.id,
      "title": event.title,
      "description": event.description,
      "start_time": event.start_time.isoformat(),
      "end_time": event.end_time.isoformat(),
  }

In [ ]:
toolset = FastMCPToolset(calendar_mcp_server)

## Ágens elkészítése

Importáljuk a szükséges modulokat, hogy létrehozzuk a naptár kezelésére szolgáló ágenst!

In [ ]:
from pydantic import BaseModel
from typing import List, Dict

import logfire

from pydantic_ai import (
    Agent,
    RunContext,
    UsageLimits,
    ModelMessage,
    RunUsage
)

In [ ]:
logfire.configure(send_to_logfire='if-token-present')
logfire.instrument_pydantic_ai()

In [ ]:
import asyncio
from pydantic_ai.exceptions import ModelHTTPError


async def run_with_rate_limit_retry(agent, prompt, attempts: int = 3, base_delay: float = 2.0):
    last_error = None
    for attempt in range(attempts):
        try:
            return await agent.run(prompt)
        except ModelHTTPError as error:
            last_error = error
            if getattr(error, 'status_code', None) != 429 or attempt == attempts - 1:
                raise
            await asyncio.sleep(base_delay * (2 ** attempt))
    raise last_error

A modell kiválasztásánál fontos figyelembe venni, hogy a modell támogassa az eszközhasználatot (tool-use).

In [ ]:
model_name = 'mistral:mistral-small-latest'

Definiáljuk a `calendar_agent` ágenst!

In [ ]:
calendar_agent = Agent(
    model_name,
    system_prompt=(
        "You are a calendar management agent whose job is to answer questions asked by users based on calendar entries. ",
        "Check the calendar for today's date."
    ),
    toolsets=[toolset]
)

Próbáljuk ki az elkészített ágensünket, tegyünk fel neki kérdéseket!

In [ ]:
result = await run_with_rate_limit_retry(calendar_agent, "What's my schedule for tomorrow?")

Amennyiben szeretnénk olvashatóbbá tenni az ágens válaszát, úgy használjuk az alábbi kódrészletet!

In [ ]:
from IPython.display import display, Markdown

In [ ]:
display(Markdown(result.output))

Kérjük meg az ágenst, hogy vegyen fel nekünk egy új naptárbejegyzést, majd utána ellenőirzzük a művelet sikerességét!

In [ ]:
result = await run_with_rate_limit_retry(calendar_agent, "Please schedule a one-hour yoga class for me tonight at 8 p.m.!")

In [ ]:
display(Markdown(result.output))

In [ ]:
result = await run_with_rate_limit_retry(calendar_agent, "What calendar entries do I have for today?")

In [ ]:
display(Markdown(result.output))

# 2. Mozi figyelő ágens

Ebben a szakaszban egy olyan MCP szervert készítünk, amely mozifilmeket és azok vetítéseit kezeli. Az ágens ezt a szervert használva tud lekérdezni információkat filmekről, valamint megtudni, hogy egyes filmeket mikor vetítenek a mozikban.

In [ ]:
from enum import Enum

class MovieGenre(str, Enum):
  thriller = 'thriller'
  action = 'action'
  adventure = 'adventure'
  drama = 'drama'
  sci_fi = 'sci-fi'
  romantic = 'romantic'
  comedy = 'comedy'

Definiáljuk a `Movie` és a `MovieScreening` adatmodelleket! A `Movie` egy mozifilmet reprezentál, amelynek a következő mezői vannak: `title`, `short_description`, `genre` és `length`. A `MovieScreening` a vetítésekhez tartozó információkat írja le, ehhez `movie_id` és `start_time` mezőket definiál.

In [ ]:
class Movie(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    title: str
    short_description: str
    genre: List[str] = Field(sa_column=Column(JSON))
    length: int
    screenings: List["MovieScreening"] = Relationship(back_populates="movie")


class MovieScreening(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    movie_id: Optional[int] = Field(default=None, foreign_key="movie.id")
    start_time: datetime
    movie: Optional[Movie] = Relationship(back_populates="screenings")

Hozzunk létre olyan adatmodelleket, amelyek általunk megszabadott módon írják le az egyes filmekhez tartozó információkat! A `MovieInfo` modell csak `id` és `title` mezőket tartalmazza, míg a `DetailedMovieInfo` modell tartalmazza továbbá a `genre`, `length` és `screenings` attributúmokat is.

In [ ]:
class MovieInfo(BaseModel):
    id: int
    title: str


class DetailedMovieInfo(MovieInfo):
    short_description: str
    genre: List[str]
    length: int
    screenings: List[datetime]

Definiáljuk az adatbázist ahol tárolni fogjuk ezeket az információkat!

In [ ]:
movie_sqlite_url = "sqlite:///movie.db"
movie_engine = create_engine(movie_sqlite_url)

In [ ]:
SQLModel.metadata.create_all(movie_engine)

## Példa adatok

Adjunk meg néhány példa filmet és vetítési alkalmat, amelyet az adatbázisunk fog tartalmazni!

In [ ]:
with Session(movie_engine) as session:
    example_movies = [
        Movie(
            title="Interstellar",
            short_description="A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival.",
            genre=["sci-fi", "adventure"],
            length=169,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 20, 9, 0)),  # Ütközik: Project Kickoff
                MovieScreening(start_time=datetime(2026, 1, 22, 18, 30)),
                MovieScreening(start_time=datetime(2026, 1, 25, 20, 0)),
                MovieScreening(start_time=datetime(2026, 1, 29, 8, 0)),   # Ütközik: Gym Workout
                MovieScreening(start_time=datetime(2026, 2, 2, 10, 0)),   # Ütközik: Strategy Planning
                MovieScreening(start_time=datetime(2026, 2, 10, 19, 0)),
                MovieScreening(start_time=datetime(2026, 2, 15, 14, 0)),
                MovieScreening(start_time=datetime(2026, 3, 1, 16, 0))
            ]
        ),
        Movie(
            title="The Dark Knight",
            short_description="When the menace known as the Joker wreaks havoc and chaos on the people of Gotham.",
            genre=["action", "thriller"],
            length=152,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 21, 10, 0)),  # Ütközik: Quick Sync
                MovieScreening(start_time=datetime(2026, 1, 24, 21, 0)),
                MovieScreening(start_time=datetime(2026, 1, 28, 15, 0)),
                MovieScreening(start_time=datetime(2026, 2, 5, 11, 0)),   # Ütközik: Technical Interview
                MovieScreening(start_time=datetime(2026, 2, 12, 16, 0)),  # Ütközik: Bug Scrub
                MovieScreening(start_time=datetime(2026, 2, 18, 19, 0)),
                MovieScreening(start_time=datetime(2026, 3, 5, 14, 0)),    # Ütközik: Product Demo
                MovieScreening(start_time=datetime(2026, 3, 12, 20, 0))
            ]
        ),
        Movie(
            title="Superbad",
            short_description="Two co-dependent high school seniors are forced to deal with separation anxiety.",
            genre=["comedy"],
            length=113,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 23, 14, 0)),  # Átfedés: Deep Work
                MovieScreening(start_time=datetime(2026, 1, 27, 18, 0)),
                MovieScreening(start_time=datetime(2026, 2, 1, 15, 0)),
                MovieScreening(start_time=datetime(2026, 2, 9, 12, 30)),  # Ütközik: Lunch with Mentor
                MovieScreening(start_time=datetime(2026, 2, 14, 20, 0)),  # Átfedés: Valentine's Dinner
                MovieScreening(start_time=datetime(2026, 2, 22, 17, 0)),
                MovieScreening(start_time=datetime(2026, 3, 3, 21, 0))
            ]
        ),
        Movie(
            title="Inception",
            short_description="A thief who steals corporate secrets through the use of dream-sharing technology.",
            genre=["sci-fi", "action"],
            length=148,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 26, 15, 0)),  # Ütközik: Client Feedback
                MovieScreening(start_time=datetime(2026, 1, 31, 20, 0)),
                MovieScreening(start_time=datetime(2026, 2, 6, 18, 0)),
                MovieScreening(start_time=datetime(2026, 2, 17, 14, 0)),  # Ütközik: Architecture Review
                MovieScreening(start_time=datetime(2026, 2, 24, 9, 0)),   # Ütközik: Workshop SQLModel
                MovieScreening(start_time=datetime(2026, 3, 6, 22, 0)),
                MovieScreening(start_time=datetime(2026, 3, 11, 10, 0))   # Átfedés: Sprint Planning
            ]
        ),
        Movie(
            title="Parasite",
            short_description="Greed and class discrimination threaten the newly formed symbiotic relationship.",
            genre=["thriller", "drama"],
            length=132,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 20, 19, 0)),
                MovieScreening(start_time=datetime(2026, 1, 24, 15, 0)),
                MovieScreening(start_time=datetime(2026, 2, 3, 20, 0)),
                MovieScreening(start_time=datetime(2026, 2, 11, 18, 0)),
                MovieScreening(start_time=datetime(2026, 2, 20, 10, 30)), # Ütközik: Coffee Break Sync
                MovieScreening(start_time=datetime(2026, 3, 2, 11, 0)),   # Ütközik: Code Review
                MovieScreening(start_time=datetime(2026, 3, 10, 19, 0)),
                MovieScreening(start_time=datetime(2026, 3, 15, 11, 0))   # Átfedés: Final Testing
            ]
        ),
        Movie(
            title="La La Land",
            short_description="While navigating their careers in Los Angeles, a pianist and an actress fall in love.",
            genre=["romantic", "drama"],
            length=128,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 22, 20, 0)),
                MovieScreening(start_time=datetime(2026, 1, 28, 19, 0)),
                MovieScreening(start_time=datetime(2026, 2, 7, 21, 0)),
                MovieScreening(start_time=datetime(2026, 2, 14, 18, 0)),  # Átfedés: Valentine's Dinner
                MovieScreening(start_time=datetime(2026, 2, 21, 15, 0)),
                MovieScreening(start_time=datetime(2026, 2, 28, 20, 0)),
                MovieScreening(start_time=datetime(2026, 3, 9, 18, 0)),   # Ütközik: Language Lesson
                MovieScreening(start_time=datetime(2026, 3, 14, 16, 0))
            ]
        ),
        Movie(
            title="The Grand Budapest Hotel",
            short_description="A writer encounters the owner of a high-class hotel, who tells him of his early years.",
            genre=["comedy", "adventure"],
            length=99,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 21, 14, 0)),
                MovieScreening(start_time=datetime(2026, 1, 25, 11, 0)),
                MovieScreening(start_time=datetime(2026, 1, 30, 17, 0)),
                MovieScreening(start_time=datetime(2026, 2, 8, 14, 0)),
                MovieScreening(start_time=datetime(2026, 2, 15, 10, 0)),
                MovieScreening(start_time=datetime(2026, 2, 23, 19, 0)),
                MovieScreening(start_time=datetime(2026, 3, 4, 18, 0)),
                MovieScreening(start_time=datetime(2026, 3, 13, 16, 0))   # Ütközik: Status Update
            ]
        ),
        Movie(
            title="Mad Max: Fury Road",
            short_description="In a post-apocalyptic wasteland, a woman rebels against a tyrannical ruler.",
            genre=["action", "adventure"],
            length=120,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 23, 10, 0)),
                MovieScreening(start_time=datetime(2026, 1, 29, 21, 0)),
                MovieScreening(start_time=datetime(2026, 2, 4, 17, 0)),
                MovieScreening(start_time=datetime(2026, 2, 12, 19, 0)),
                MovieScreening(start_time=datetime(2026, 2, 19, 22, 0)),
                MovieScreening(start_time=datetime(2026, 2, 27, 15, 0)),  # Ütközik: Market Research
                MovieScreening(start_time=datetime(2026, 3, 8, 14, 0)),
                MovieScreening(start_time=datetime(2026, 3, 14, 21, 0))
            ]
        ),
        Movie(
            title="Dune",
            short_description="A noble family becomes embroiled in a war for control over the galaxy's most valuable asset.",
            genre=["sci-fi", "adventure"],
            length=155,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 24, 10, 0)),
                MovieScreening(start_time=datetime(2026, 1, 31, 14, 0)),
                MovieScreening(start_time=datetime(2026, 2, 10, 15, 0)),
                MovieScreening(start_time=datetime(2026, 2, 18, 20, 0)),
                MovieScreening(start_time=datetime(2026, 2, 25, 17, 0)),
                MovieScreening(start_time=datetime(2026, 3, 7, 21, 0)),
                MovieScreening(start_time=datetime(2026, 3, 15, 9, 0))    # Átfedés: Final Testing
            ]
        ),
        Movie(
            title="Shutter Island",
            short_description="A U.S. Marshal investigates the disappearance of a murderer who escaped from a hospital.",
            genre=["thriller", "drama"],
            length=138,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 20, 22, 0)),
                MovieScreening(start_time=datetime(2026, 1, 27, 20, 0)),
                MovieScreening(start_time=datetime(2026, 2, 2, 18, 0)),
                MovieScreening(start_time=datetime(2026, 2, 11, 21, 0)),
                MovieScreening(start_time=datetime(2026, 2, 21, 20, 0)),
                MovieScreening(start_time=datetime(2026, 3, 1, 14, 0)),
                MovieScreening(start_time=datetime(2026, 3, 8, 19, 0)),
                MovieScreening(start_time=datetime(2026, 3, 12, 17, 0))
            ]
        )
    ]
    session.add_all(example_movies)
    session.commit()

## MCP szerver elkészítése

Definiáljuk a `Movie` nevű MCP szerverünket, illetve készítsük el hozzá a szükséges eszközöket!

In [ ]:
movie_mcp_server = FastMCP('Movie')

Definiáljuk a `get_title_of_movies` eszközt, amely kilistázza az adatbázisban tárolt összes film címét!

In [ ]:
@movie_mcp_server.tool()
def get_title_of_movies():
  with Session(movie_engine) as session:
      movies = session.exec(select(Movie).order_by(Movie.title)).all()
  return [movie.title for movie in movies]

Definiáljuk a `get_screenings_for_movie` eszközt, amely egy konkrét film címéhez kikeresi az egyes vetítési alkalmait!

In [ ]:
@movie_mcp_server.tool()
def get_screenings_for_movie(movie_title: str):
  with Session(movie_engine) as session:
      movie = session.exec(select(Movie).options(selectinload(Movie.screenings)).where(Movie.title == movie_title)).first()

  if movie is None:
      return {"error": f"Movie not found: {movie_title}"}

  return {
      "movie_id": movie.id,
      "title": movie.title,
      "screenings": [screening.start_time.isoformat() for screening in movie.screenings],
  }

Definiáljuk a `get_movie_info` eszközt, amely egy konkrét id-val rendelkező filmhez lekérdezi az összes információt!

In [ ]:
@movie_mcp_server.tool()
def get_movie_info(movie_id: str) -> DetailedMovieInfo:
    with Session(movie_engine) as session:
        movie = session.exec(select(Movie).options(selectinload(Movie.screenings)).where(Movie.id == int(movie_id))).first()

    if movie is None:
        raise ValueError(f"Movie not found: {movie_id}")

    return DetailedMovieInfo(
        id=movie.id,
        title=movie.title,
        short_description=movie.short_description,
        genre=movie.genre,
        length=movie.length,
        screenings=[screening.start_time for screening in movie.screenings],
    )

Definiáljunk egy `search_movies_by_filter` eszközt, amely összetett lekérdezéseket tesz lehetővé! A `genres` paraméterében opcionálisan megadható egy műfaj lista (`or` művelettel összefűzve a feltételben), illetve a `target_date` paraméterében opcionálisan megadható dátum. A `response_format` lehetővé teszi, hogy az ágens válasza a kapott válasz részletezettségét.

In [ ]:
@movie_mcp_server.tool()
def search_movies_by_filter(
    genres: Optional[List[Literal['thriller', 'action', 'adventure', 'drama', 'sci-fi', 'romantic', 'comedy']]] = None,
    target_date: Optional[datetime] = None,
    response_format: Literal['concise', 'detailed'] = "detailed"
    ) -> Union[List[MovieInfo], List[DetailedMovieInfo]]:
  with Session(movie_engine) as session:
    movies = session.exec(select(Movie).options(selectinload(Movie.screenings))).all()

  if target_date:
    target_day = target_date.date()
    movies = [
        movie
        for movie in movies
        if any(screening.start_time.date() == target_day for screening in movie.screenings)
    ]

  if genres:
    movies = [movie for movie in movies if any(genre in movie.genre for genre in genres)]

  if response_format == "concise":
    return [MovieInfo(title=movie.title, id=movie.id) for movie in movies]

  return [DetailedMovieInfo(
      title=movie.title,
      id=movie.id,
      short_description=movie.short_description,
      genre=movie.genre,
      length=movie.length,
      screenings=[screening.start_time for screening in movie.screenings]
  ) for movie in movies]

Próbáljuk ki az elkészített `search_movies_by_filter` működését!

In [ ]:
search_movies_by_filter(target_date=datetime(2026, 1, 20))

In [ ]:
get_screenings_for_movie("Interstellar")

In [ ]:
get_movie_info("1")

## Ágens elkészítése

Készítsük el a mozi kereső ágensünket!

In [ ]:
toolset = FastMCPToolset(movie_mcp_server)

In [ ]:
movie_agent = Agent(
    model_name,
    system_prompt=(
        "You are a movie search agent who can provide information about movies that meet the user's request. ",
    ),
    toolsets=[toolset]
)

## Ágens tesztelése

A következő szakaszban készítsünk pár példa hívást, amivel megvizsgálhatjuk mennyire jól működik együtt az ágens az általunk készített MCP szerverrel.

### 1. eset: "What are the names of the movies?"

In [ ]:
result = await run_with_rate_limit_retry(movie_agent, "What are the names of the movies?")

In [ ]:
display(Markdown(result.output))

In [ ]:
get_title_of_movies()

### 2. eset: "What adventure movies are showing at the moment?"

In [ ]:
result = await run_with_rate_limit_retry(movie_agent, "What adventure movies are showing at the moment?")

In [ ]:
display(Markdown(result.output))

### 3. eset: "Recommend some movies for me for the afternoon of March 7, 2026."

In [ ]:
result = await run_with_rate_limit_retry(movie_agent, "Recommend some movies for me for the afternoon of March 7, 2026.")

In [ ]:
display(Markdown(result.output))

In [ ]:
search_movies_by_filter( target_date=datetime(2026, 3, 7))

## Saját próbálkozások

Írjatok még néhány kérést az ágens számára (esetleg további eszközöket definiálva) és vizsgáljátok meg miképp viselkedik az ágens!

In [ ]:
result = await run_with_rate_limit_retry(movie_agent, "List the available movies and suggest one adventure movie for this week.")
display(Markdown(result.output))

# 3. Szorgalmi: MCP szerverek kombinálása – Kooperatív megközelítés

## Szorgalmi feladat leírása

Készíts egy olyan megoldást, amely az elkészített `calendar_agent` és `movie_agent` segítségével képes:
1. **Filmet ajánlani** a felhasználó preferenciái alapján
2. **Ellenőrizni** a naptárat az ütközésekre
3. **Automatikusan ütemezni** az ajánlott filmet a naptárba

## Megközelítés: Kooperatív ágensek (Sequential Method)

**Elv**: Az ágensek egymás után futnak, az egyik kimenetét a másik inputjaként használjuk.

In [ ]:
## Megközelítés 1: Kooperatív ágensek (Sequential Method)

**Elv**: Az ágensek egymás után futnak, az egyik kimenetét a másik inputjaként használjuk.

import json
from typing import Any, Dict, Optional


def _extract_json_object(text: str) -> str:
    """JSON objektum kinyerése szövegből."""
    start_index = text.find("{")
    end_index = text.rfind("}")
    if start_index == -1 or end_index == -1 or end_index <= start_index:
        raise ValueError("The movie agent did not return a JSON object.")
    return text[start_index : end_index + 1]


async def recommend_and_schedule_movie(
    preferences: str,
    preferred_date: Optional[datetime] = None,
    calendar_event_title: str = "Movie night",
) -> Dict[str, Any]:
    """
    Kooperatív megközelítés: movie_agent és calendar_agent szekvenciális hívása.
    
    1. lépés: movie_agent ajánlást ad (JSON formátumban)
    2. lépés: calendar_agent beütemezi az eseményt
    
    Args:
        preferences: Film ajánlásra vonatkozó felhasználói igény
        preferred_date: Preferált dátum (opcionális)
        calendar_event_title: A naptári esemény címe
    
    Returns:
        Dict: A film ajánlást és a naptár szervezés eredményét tartalmazó dict
    """
    # 1. LÉPÉS: Movie agent ajánlást ad
    movie_prompt = (
        "Use the movie tools to recommend one movie that best matches the user's request. "
        "Return ONLY valid JSON with the keys movie_title, screening_start, screening_end, and reason. "
        "Use ISO 8601 strings for the datetimes. "
        f"User request: {preferences}."
    )
    if preferred_date is not None:
        movie_prompt += f" Preferred date: {preferred_date.date().isoformat()}."

    movie_result = await run_with_rate_limit_retry(movie_agent, movie_prompt)
    recommendation = json.loads(_extract_json_object(movie_result.output))

    # 2. LÉPÉS: Calendar agent beütemezi az eseményt
    calendar_prompt = (
        f"Schedule a calendar event with title '{calendar_event_title}: {recommendation['movie_title']}', "
        f"description '{recommendation['reason']}', "
        f"start_time '{recommendation['screening_start']}', "
        f"end_time '{recommendation['screening_end']}'. "
        "If the time slot is busy, do not insert the event and explain the conflict briefly."
    )
    calendar_result = await run_with_rate_limit_retry(calendar_agent, calendar_prompt)

    return {
        "movie_recommendation": recommendation,
        "calendar_result": calendar_result.output,
    }

In [ ]:
### Teszt 1: Sci-fi film ajánlása február 15-én

result = await recommend_and_schedule_movie(
    preferences="Recommend a sci-fi movie for me that's not too long.",
    preferred_date=datetime(2026, 2, 15),
    calendar_event_title="Movie night"
)

print("🎬 FILM AJÁNLÁS:")
print(json.dumps(result["movie_recommendation"], indent=2))
print("\n📅 NAPTÁR ÜTEMEZÉS EREDMÉNYE:")
print(result["calendar_result"])

In [ ]:
### Teszt 2: Adventure film ajánlása

result = await recommend_and_schedule_movie(
    preferences="I want to watch an adventure movie this week that I'm free for.",
    preferred_date=datetime(2026, 1, 25),
    calendar_event_title="Adventure Cinema Night"
)

print("🎬 FILM AJÁNLÁS:")
print(json.dumps(result["movie_recommendation"], indent=2))
print("\n📅 NAPTÁR ÜTEMEZÉS EREDMÉNYE:")
print(result["calendar_result"])